# Kaggle Submission Pipeline
This notebook generates `submission.csv` using the chosen configuration. It mirrors the robust logic and parsing of `run_models.ipynb`.

In [1]:
# === CLEAR STALE MODULES ===
import sys
import importlib
import os

for mod in list(sys.modules.keys()):
	if 'configuration_slimmoe' in mod or 'modeling_slimmoe' in mod:
		del sys.modules[mod]

importlib.invalidate_caches()
print("Stale modules cleared. Please restart the kernel once more and run your model loading cell.")

Stale modules cleared. Please restart the kernel once more and run your model loading cell.


In [2]:
import json
import pandas as pd
from tqdm import tqdm
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Glm4vForConditionalGeneration
import os
import string
import re

In [3]:
# [0]=Qwen, [1]=Llama, [2]=Gemma, [3]=GLM, [4]=Ministral, [5]=Phi
SELECTED_INDEX = 2
PROMPT_TYPE = "zero_shot" # Options: "zero_shot", "cot", "two_shot", "four_shot"

USE_POSITION_DEBIASING = True
USE_SELF_CONSISTENCY = False

In [4]:
# Parameters
SELECTED_INDEX = 3
PROMPT_TYPE = "zero_shot"
USE_POSITION_DEBIASING = True
USE_SELF_CONSISTENCY = False


In [5]:
# === EXPERIMENT CONFIGURATION ===
MODEL_OPTIONS = [
    ("models/qwen2.5-7b-instruct", "qwen2.5-7b-instruct"),
    ("models/llama3.1-8b-instruct", "llama3.1-8b-instruct"),
    ("models/gemma2-9b-it", "gemma2-9b-it"),
    ("models/glm4.1v-9b-thinking", "glm4.1v-9b-thinking")
]

SC_TEMPERATURES = [0.0, 0.1, 0.2]

MAX_TOKENS = 4096 if "thinking" in MODEL_OPTIONS[SELECTED_INDEX][1].lower() else 1024

MODEL_ID, MODEL_NAME = MODEL_OPTIONS[SELECTED_INDEX]
TEST_DATA_PATH = "data/test.json"

suffix = ""
if USE_SELF_CONSISTENCY and USE_POSITION_DEBIASING: suffix = "_sc_pd"
elif USE_SELF_CONSISTENCY: suffix = "_sc"
elif USE_POSITION_DEBIASING: suffix = "_pd"
else: suffix = "_baseline"

SUBMISSION_CSV = f"submissions/submission_{MODEL_NAME}_{PROMPT_TYPE}{suffix}.csv"

print(f"ACTIVE RUN")
print(f"{'-'*30}")
print(f"Model Name:  {MODEL_NAME}")
print(f"Model Path:  {MODEL_ID}")
print(f"Prompt:      {PROMPT_TYPE}")
print(f"Pos Debias:  {USE_POSITION_DEBIASING}")
print(f"Self-Consis: {USE_SELF_CONSISTENCY}")
print(f"Max Tokens:  {MAX_TOKENS}")
print(f"{'-'*30}")

ACTIVE RUN
------------------------------
Model Name:  glm4.1v-9b-thinking
Model Path:  models/glm4.1v-9b-thinking
Prompt:      zero_shot
Pos Debias:  True
Self-Consis: False
Max Tokens:  4096
------------------------------


In [6]:
# === LOADING TOKENIZER AND MODEL TO GPU ===
print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print("Loading Model to GPU...")
if MODEL_NAME == "ministral-3-8b-instruct-2512-bf16":
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, 
        quantization_config=quant_config,
        device_map="auto", 
        trust_remote_code=True
    )
elif MODEL_NAME == "phi-mini-MoE-instruct":
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, 
        quantization_config=quant_config,
        device_map="auto", 
        trust_remote_code=True
    )
elif MODEL_NAME == "gemma2-9b-it":
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, 
        quantization_config=quant_config,
        device_map="auto", 
        trust_remote_code=True
    )
elif MODEL_NAME == "glm4.1v-9b-thinking":
    model = Glm4vForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, 
        device_map="auto", 
        torch_dtype=torch.bfloat16,
        trust_remote_code=True
)

# Set up the inferencer (Pipeline or Direct Model)
if MODEL_NAME == "glm4.1v-9b-thinking":
    inferencer = model
else:
    inferencer = pipeline(
        "text-generation", 
        model=model, 
        tokenizer=tokenizer,
        return_full_text=False
    )

print("✅ Pipeline Ready!")


Loading Tokenizer...


Loading Model to GPU...


Loading weights:   0%|          | 0/704 [00:00<?, ?it/s]

✅ Pipeline Ready!


In [7]:
# === DEFINE PROMPTS ===
def format_conversation(dialog):
	return "\n".join([f"{msg['role'].capitalize()}: {msg['content']}" for msg in dialog])

def create_zero_shot_prompt(dialog_1, dialog_2):
	d1_text = format_conversation(dialog_1)
	d2_text = format_conversation(dialog_2)
	return f"""
	[System Instructions]
	You are an expert evaluator assessing the quality of two AI responses to the same user question. 
	Your goal is to determine which response better meets human preferences based on accuracy, helpfulness, and clarity.
	[Dialog 1]
	{d1_text}
	[Dialog 2]
	{d2_text}
	[Task]
	Evaluate both dialogs. Output your final verdict EXACTLY as one of these 4 words and nothing else:
	- "A" if Dialog A is noticeably better.
	- "B" if Dialog B is noticeably better.
	- "tie" if both are of similar quality (good or average).
	- "neither" if both are completely unhelpful, irrelevant, or dangerously wrong.
	Verdict:
	"""

def create_cot_prompt(dialog_1, dialog_2):
	d1_text = format_conversation(dialog_1)
	d2_text = format_conversation(dialog_2)
	return f"""
	[System Instructions]
	You are an expert evaluator assessing the quality of two AI responses to the same user question. 
	Your goal is to determine which response better meets human preferences based on accuracy, helpfulness, and clarity.
	[Dialog 1]
	{d1_text}
	[Dialog 2]
	{d2_text}
	[Task]
	First, write a brief, step-by-step critique comparing the two dialogs on helpfulness, relevance, and clarity. 
	Then, using that critique that you just wrote, on a new line at the very end, provide your final verdict EXACTLY as one of these 4 words: "A", "B", "tie", or "neither".
	- "A" if Dialog A is noticeably better.
	- "B" if Dialog B is noticeably better.
	- "tie" if both are of similar quality (good or average).
	- "neither" if both are completely unhelpful, irrelevant, or dangerously wrong.
	Critique and Verdict:
	"""

def create_two_shot_prompt(dialog_1, dialog_2):
	d1_text = format_conversation(dialog_1)
	d2_text = format_conversation(dialog_2)
	return f"""
	[System Instructions]
	You are an expert evaluator assessing the quality of two AI responses to the same user question. 
	Your goal is to determine which response better meets human preferences based on accuracy, helpfulness, and clarity.
	Here are two examples of how you should evaluate.
	--- Example 1 ---
	[Dialog 1]
	User: Explain why some people think that humans never landed on the moon
	Assistant:Some people believe that humans never landed on the moon for several reasons. Some of these reasons include:\n1. Lack of evidence: There is limited evidence to support the idea that humans have actually landed on the moon, such as satellite photos, radar imagery, and other forms of evidence. These sources may not have been widely used or analyzed, or they may have been viewed as inadequate or inaccurate.\n2. Confidence bias: Some people may have a strong belief in the idea that humans have landed on the moon and may be skeptical or have a lack of faith in the scientific evidence supporting the mission.\n3. Personal beliefs: Some people may hold beliefs that are not supported by science, such as conspiracy theory or skepticism. These beliefs may be based on personal experiences, beliefs, or attitudes, and may not be based on evidence or reasoning.\n4. Personal experiences: Some people may have personal experiences or memories that are not supported by scientific evidence, which may lead them to believe that humans never landed on the moon.\nIt's important to note that the evidence for the existence of humans on the moon is overwhelming, and the evidence for the moon's existence is based on a wide range of sources, including scientific evidence, satellite images, and personal experiences. Despite these overwhelming facts, some people may still hold beliefs that are not supported",
	[Dialog 2]
	User: Explain why some people think that humans never landed on the moon
	Assistant:There are several reasons, including that the United States government spent a large amount of money on the space program, and did not want to advertise that we did not have a manned mission to the moon a success.  There were other national security concerns as well.  Also, the technology at the time did not exist to support a manned mission to the lunar surface.  Finally, the Apollo missions intentionally flew orbits that took them away from the moon, to prevent contamination from Earth organisms, and to avoid accidentally crashing the spacecraft on the moon.  The United States did return samples from the lunar surface, but those samples are in a lab on Earth, and are not on the moon.",
	Verdict: B
	--- Example 2 ---
	[Dialog 1]
	User: what isomer is the alkene in C/C=C/c2ccc(COc1cccc(CCO)c1)cc2?
	Assistant: The alkene in C/C=C/c2ccc(COc1cccc(CCO)c1)cc2 is a trans-alkene. Its formula is R1C(=CCCC(=CC(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(C(=O)OC(C(C(C(=O)OC(C(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(=O)OC(C(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(=O)OC(C(C(C(=O)OC(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(=O)OC(C(C(C(C(=O)OC(C(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(C(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(C(C(=",
	[Dialog 2]
	User: what isomer is the alkene in C/C=C/c2ccc(COc1cccc(CCO)c1)cc2?
	Assistant: The alkene in the given molecule is 1-butene, which is a straight-chain alkene with the molecular formula C4H8. The double bond in 1-butene is between the first and second carbon atoms.",
	Verdict: A
	------------------------
	[Task]
	Now evaluate the following dialogs. Output your final verdict EXACTLY as one of these 4 words and nothing else: "A", "B", "tie", or "neither".
	- "A" if Dialog A is noticeably better.
	- "B" if Dialog B is noticeably better.
	- "tie" if both are of similar quality (good or average).
	- "neither" if both are completely unhelpful, irrelevant, or dangerously wrong.

	[Dialog A]
	{d1_text}
	[Dialog B]
	{d2_text}
	Verdict:
	"""

def create_four_shot_prompt(dialog_1, dialog_2):
	d1_text = format_conversation(dialog_1)
	d2_text = format_conversation(dialog_2)
	return f"""
	[System Instructions]
	You are an expert evaluator assessing the quality of two AI responses to the same user question. 
	Your goal is to determine which response better meets human preferences based on accuracy, helpfulness, and clarity.
	Here are four examples of how you should evaluate.
	--- Example 1 ---
	[Dialog 1]
	User: Explain why some people think that humans never landed on the moon
	Assistant:Some people believe that humans never landed on the moon for several reasons. Some of these reasons include:\n1. Lack of evidence: There is limited evidence to support the idea that humans have actually landed on the moon, such as satellite photos, radar imagery, and other some form of evidence. These sources may not have been widely used or analyzed, or they may have been viewed as inadequate or inaccurate.\n2. Confidence bias: Some people may have a strong belief in the idea that humans have landed on the moon and may be skeptical or have a lack of faith in the scientific evidence supporting the mission.\n3. Personal beliefs: Some people may hold beliefs that are not supported by science, such as conspiracy theory or skepticism. These beliefs may be based on personal experiences, beliefs, or attitudes, and may not be based on evidence or reasoning.\n4. Personal experiences: Some people may have personal experiences or memories that are not supported by scientific evidence, which may lead them to believe that humans never landed on the moon.\nIt's important to note that the evidence for the existence of humans on the moon is overwhelming, and the evidence for the moon's existence is based on a wide range of sources, including scientific evidence, satellite images, and personal experiences. Despite these overwhelming facts, some people may still hold beliefs that are not supported",
	[Dialog 2]
	User: Explain why some people think that humans never landed on the moon
	Assistant:There are several reasons, including that the United States government spent a large amount of money on the space program, and did not want to advertise that we did not have a manned mission to the moon a success.  There were other national security concerns as well.  Also, the technology at the time did not exist to support a manned mission to the lunar surface.  Finally, the Apollo missions intentionally flew orbits that took them away from the moon, to prevent contamination from Earth organisms, and to avoid accidentally crashing the spacecraft on the moon.  The United States did return samples from the lunar surface, but those samples are in a lab on Earth, and are not on the moon.",
	Verdict: B
	--- Example 2 ---
	[Dialog A]
	User: How many words are in your next response?
	Assistant: 15
	[Dialog B]
	User: How many words are in your next response?
	Assistant: My next response will contain approximately 4 words.
	Verdict: neither
	--- Example 3 ---
	[Dialog 1]
	User: what isomer is the alkene in C/C=C/c2ccc(COc1cccc(CCO)c1)cc2?
	Assistant: The alkene in C/C=C/c2ccc(COc1cccc(CCO)c1)cc2 is a trans-alkene. Its formula is R1C(=CCCC(=CC(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(C(=O)OC(C(C(C(=O)OC(C(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(=O)OC(C(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(=O)OC(C(C(C(=O)OC(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(=O)OC(C(C(C(C(=O)OC(C(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(C(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(C(C(=",
	[Dialog 2]
	User: what isomer is the alkene in C/C=C/c2ccc(COc1cccc(CCO)c1)cc2?
	Assistant: The alkene in the given molecule is 1-butene, which is a straight-chain alkene with the molecular formula C4H8. The double bond in 1-butene is between the first and second carbon atoms.",
	Verdict: A
	--- Example 4 ---
	[Dialog 1]
	User: Was Elvis an alien?
	Assistant: As an AI language model, I do not have personal beliefs or opinions. However, there is no conclusive evidence to suggest that Elvis was an alien or that he kept his true identity hidden. Some fans have speculated that he may have been abducted by aliens, but there is no concrete proof. Elvis' life and career have been the subject of much speculation and rumor, but it is important to remember that the truth may never be known.
	[Dialog 2]
	User: Was Elvis an alien?
	Assistant: There is no evidence to support the claim that Elvis Presley was an alien. Elvis was a famous singer, actor, and musician who was born and raised in the United States.
	Verdict: tie
	------------------------
	[Task]
	Now evaluate the following dialogs. Output your final verdict EXACTLY as one of these 4 words and nothing else: "A", "B", "tie", or "neither".
	- "A" if Dialog A is noticeably better.
	- "B" if Dialog B is noticeably better.
	- "tie" if both are of similar quality (good or average).
	- "neither" if both are completely unhelpful, irrelevant, or dangerously wrong.

	[Dialog A]
	{d1_text}
	[Dialog B]
	{d2_text}
	Verdict:
	"""

# def create_four_shot_cot_prompt(dialog_1, dialog_2):
# 	d1_text = format_conversation(dialog_1)
# 	d2_text = format_conversation(dialog_2)
# 	return f"""
# 	[System Instructions]
# 	You are an expert evaluator assessing the quality of two AI responses to the same user question. 
# 	Your goal is to determine which response better meets human preferences based on accuracy, helpfulness, and clarity.
# 	Here are four examples of how you should evaluate.
# 	--- Example 1 ---
# 	[Dialog 1]
# 	User: Explain why some people think that humans never landed on the moon
# 	Assistant:Some people believe that humans never landed on the moon for several reasons. Some of these reasons include:\n1. Lack of evidence: There is limited evidence to support the idea that humans have actually landed on the moon, such as satellite photos, radar imagery, and other some form of evidence. These sources may not have been widely used or analyzed, or they may have been viewed as inadequate or inaccurate.\n2. Confidence bias: Some people may have a strong belief in the idea that humans have landed on the moon and may be skeptical or have a lack of faith in the scientific evidence supporting the mission.\n3. Personal beliefs: Some people may hold beliefs that are not supported by science, such as conspiracy theory or skepticism. These beliefs may be based on personal experiences, beliefs, or attitudes, and may not be based on evidence or reasoning.\n4. Personal experiences: Some people may have personal experiences or memories that are not supported by scientific evidence, which may lead them to believe that humans never landed on the moon.\nIt's important to note that the evidence for the existence of humans on the moon is overwhelming, and the evidence for the moon's existence is based on a wide range of sources, including scientific evidence, satellite images, and personal experiences. Despite these overwhelming facts, some people may still hold beliefs that are not supported",
# 	[Dialog 2]
# 	User: Explain why some people think that humans never landed on the moon
# 	Assistant:There are several reasons, including that the United States government spent a large amount of money on the space program, and did not want to advertise that we did not have a manned mission to the moon a success.  There were other national security concerns as well.  Also, the technology at the time did not exist to support a manned mission to the lunar surface.  Finally, the Apollo missions intentionally flew orbits that took them away from the moon, to prevent contamination from Earth organisms, and to avoid accidentally crashing the spacecraft on the moon.  The United States did return samples from the lunar surface, but those samples are in a lab on Earth, and are not on the moon.",
# 	Verdict: B
# 	--- Example 2 ---
# 	[Dialog A]
# 	User: How many words are in your next response?
# 	Assistant: 15
# 	[Dialog B]
# 	User: How many words are in your next response?
# 	Assistant: My next response will contain approximately 4 words.
# 	Verdict: neither
# 	--- Example 3 ---
# 	[Dialog 1]
# 	User: what isomer is the alkene in C/C=C/c2ccc(COc1cccc(CCO)c1)cc2?
# 	Assistant: The alkene in C/C=C/c2ccc(COc1cccc(CCO)c1)cc2 is a trans-alkene. Its formula is R1C(=CCCC(=CC(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(C(=O)OC(C(C(C(=O)OC(C(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(=O)OC(C(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(=O)OC(C(C(C(=O)OC(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(=O)OC(C(C(C(C(=O)OC(C(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(C(C(C(=O)OC(C(C(C(C(C(C(C(=O)OC(C(C(C(C(C(=O)OC(C(C(C(C(C(=",
# 	[Dialog 2]
# 	User: what isomer is the alkene in C/C=C/c2ccc(COc1cccc(CCO)c1)cc2?
# 	Assistant: The alkene in the given molecule is 1-butene, which is a straight-chain alkene with the molecular formula C4H8. The double bond in 1-butene is between the first and second carbon atoms.",
# 	Verdict: A
# 	--- Example 4 ---
# 	[Dialog 1]
# 	User: Was Elvis an alien?
# 	Assistant: As an AI language model, I do not have personal beliefs or opinions. However, there is no conclusive evidence to suggest that Elvis was an alien or that he kept his true identity hidden. Some fans have speculated that he may have been abducted by aliens, but there is no concrete proof. Elvis' life and career have been the subject of much speculation and rumor, but it is important to remember that the truth may never be known.
# 	[Dialog 2]
# 	User: Was Elvis an alien?
# 	Assistant: There is no evidence to support the claim that Elvis Presley was an alien. Elvis was a famous singer, actor, and musician who was born and raised in the United States.
# 	Verdict: tie
# 	------------------------
# 	[Task]
# 	First, write a brief, step-by-step critique comparing the two dialogs on helpfulness, relevance, and clarity. 
# 	Then, using that critique that you just wrote, on a new line at the very end, provide your final verdict EXACTLY as one of these 4 words: "A", "B", "tie", or "neither".
# 	- "A" if Dialog A is noticeably better.
# 	- "B" if Dialog B is noticeably better.
# 	- "tie" if both are of similar quality (good or average).
# 	- "neither" if both are completely unhelpful, irrelevant, or dangerously wrong.
# 	Critique and Verdict:
# 	"""

def create_prompt(dialog_1, dialog_2, prompt_type):
	if prompt_type == "zero_shot": 
		return create_zero_shot_prompt(dialog_1, dialog_2)
	elif prompt_type == "cot": 
		return create_cot_prompt(dialog_1, dialog_2)
	elif prompt_type == "two_shot": 
		return create_two_shot_prompt(dialog_1, dialog_2)
	elif prompt_type == "four_shot": 
		return create_four_shot_prompt(dialog_1, dialog_2)
	# elif prompt_type == "four_shot_cot": 
	# 	return create_four_shot_cot_prompt(dialog_1, dialog_2)

print("✅ All Prompt Functions defined!")


✅ All Prompt Functions defined!


In [8]:
# === INFERENCE & PARSER ===
def generate_text(prompt, max_tokens=MAX_TOKENS, temperature=0.0):
    system_unfriendly_models = ["gemma", "phi", "ministral", "glm"]
    if any(m in MODEL_NAME.lower() for m in system_unfriendly_models):
        messages = [{"role": "user", "content": "System Instructions: You are a helpful evaluator.\n\n" + prompt}]
    else:
        messages = [{"role": "system", "content": "You are a helpful evaluator."}, {"role": "user", "content": prompt}]

    if MODEL_NAME == "glm4.1v-9b-thinking":
        prompt_str = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        inputs = tokenizer(prompt_str, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=max_tokens, temperature=temperature, do_sample=(temperature > 0), pad_token_id=tokenizer.eos_token_id)
        input_len = inputs['input_ids'].shape[1]
        return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    else:
        outputs = inferencer(messages, max_new_tokens=max_tokens, temperature=temperature, do_sample=(temperature > 0))
        return outputs[0]["generated_text"].strip()

# def parse_verdict(response_text, prompt_type):
#     import re
#     import string

#     # 1. Remove Thinking Tags first
#     clean_text = response_text
#     if "</think>" in response_text:
#         clean_text = response_text.split("</think>")[-1]
    
#     # 2. Strip ALL other HTML tags (like <answer>, <div>, etc.)
#     clean_text = re.sub(r'<[^>]+>', '', clean_text).strip()
    
#     # 3. Pattern Search (Highest Reliability)
#     # Looks for "Verdict: A", "Answer: B", etc. anywhere in the cleaned text
#     verdict_match = re.search(r'(?:Verdict|Answer|Result|Choice):\s*([AB]|tie|neither)', clean_text, re.IGNORECASE)
#     if verdict_match:
#         val = verdict_match.group(1).lower()
#         return (val.upper() if val in ["a", "b"] else val), response_text

#     # 4. Line-based fallback
#     lines = [l.strip() for l in clean_text.split("\n") if l.strip()]
#     if not lines: return "tie", response_text
    
#     # Check first and last lines of the remaining text
#     for cand_line in [lines[0], lines[-1]]:
#         c_clean = cand_line.lower().translate(str.maketrans('', '', string.punctuation))
#         words = c_clean.split()
#         if words:
#             # If the line starts with A/B/tie/neither
#             if words[0] in ["a", "b", "tie", "neither"]:
#                 val = words[0]
#                 return (val.upper() if val in ["a", "b"] else val), response_text
#             # If the line is "The winner is A" or similar
#             if "is a" in c_clean or "better a" in c_clean: return "A", response_text
#             if "is b" in c_clean or "better b" in c_clean: return "B", response_text

#     return "tie", response_text

def parse_verdict(response_text, prompt_type):
    import re
    import string

    # 1. Start with the raw response
    clean_text = response_text
    
    # 2. Remove DeepSeek/GLM thinking tags
    if "</think>" in clean_text:
        clean_text = clean_text.split("</think>")[-1]
    
    # 3. Strip ALL HTML-like tags (like <answer>, <div>, etc.)
    # This prevents the parser from seeing "A</answer>" as "Aanswer"
    clean_text = re.sub(r'<[^>]+>', '', clean_text).strip()
    
    # 4. Keyword Search (highest priority)
    # Searches for "Verdict: A", "Answer: B", "Choice: tie", etc.
    verdict_match = re.search(r'(?:Verdict|Answer|Result|Choice):\s*([AB]|tie|neither)', clean_text, re.IGNORECASE)
    if verdict_match:
        val = verdict_match.group(1).lower()
        return (val.upper() if val in ["a", "b"] else val), response_text

    # 5. Line-based Fallback
    lines = [l.strip() for l in clean_text.split("\n") if l.strip()]
    if not lines: 
        return "tie", response_text
    
    # Check the first and last lines for a clear verdict
    for cand_line in [lines[0], lines[-1]]:
        # Remove punctuation and check the first word
        line_clean = cand_line.lower().translate(str.maketrans('', '', string.punctuation))
        words = line_clean.split()
        
        if words:
            # Case 1: The line starts with the verdict (e.g. "A because...")
            if words[0] in ["a", "b", "tie", "neither"]:
                val = words[0]
                return (val.upper() if val in ["a", "b"] else val), response_text
            
            # Case 2: The line contains a winning phrase (e.g. "Dialog A is better")
            if "is a" in line_clean or "better a" in line_clean or "dialog a" in line_clean:
                return "A", response_text
            if "is b" in line_clean or "better b" in line_clean or "dialog b" in line_clean:
                return "B", response_text

    return "tie", response_text

def get_voted_prediction(prompt, p_type, max_tokens):
    if not USE_SELF_CONSISTENCY:
        raw = generate_text(prompt, max_tokens=max_tokens, temperature=0.0)
        return parse_verdict(raw, p_type)
    
    votes = []
    last_response = ""
    for temp in SC_TEMPERATURES:
        raw = generate_text(prompt, max_tokens=max_tokens, temperature=temp)
        pred, resp = parse_verdict(raw, p_type)
        votes.append(pred)
        last_response = resp
    
    voted_pred = max(set(votes), key=votes.count)
    return voted_pred, last_response

print("✅ Generator, Parser, and Self-Consistency ready!")


✅ Generator, Parser, and Self-Consistency ready!


In [9]:
with open(TEST_DATA_PATH, "r") as f:
    test_data = json.load(f)

results = []
# NOTE:
for item in tqdm(test_data, desc="Kaggle Inference"):
# for item in tqdm(test_data[:20], desc="Kaggle Inference"):
    try:
        prompt_1 = create_prompt(item["dialog_1"], item["dialog_2"], PROMPT_TYPE)
        pred_1, _ = get_voted_prediction(prompt_1, PROMPT_TYPE, MAX_TOKENS)
        prediction = pred_1
        
        if USE_POSITION_DEBIASING:
            prompt_2 = create_prompt(item["dialog_2"], item["dialog_1"], PROMPT_TYPE)
            pred_2, _ = get_voted_prediction(prompt_2, PROMPT_TYPE, MAX_TOKENS)
            if pred_1 == "A" and pred_2 == "B": prediction = "A"
            elif pred_1 == "B" and pred_2 == "A": prediction = "B"
            else: prediction = "tie"
            
    except Exception as e:
        print(f"Error on {item['id']}: {e}")
        prediction = "tie"
    
    results.append({"id": item["id"], "verdict": prediction})

df_sub = pd.DataFrame(results)
df_sub.to_csv(SUBMISSION_CSV, index=False)
print(f"✅ Saved to {SUBMISSION_CSV}")

Kaggle Inference:   0%|          | 0/1000 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Kaggle Inference:   0%|          | 1/1000 [00:15<4:24:43, 15.90s/it]

Kaggle Inference:   0%|          | 2/1000 [00:24<3:12:04, 11.55s/it]

Kaggle Inference:   0%|          | 3/1000 [01:12<7:50:59, 28.34s/it]

...

Kaggle Inference: 100%|█████████▉| 998/1000 [7:18:14<00:21, 10.54s/it]

Kaggle Inference: 100%|█████████▉| 999/1000 [7:18:53<00:19, 19.02s/it]

Kaggle Inference: 100%|██████████| 1000/1000 [7:19:04<00:00, 16.64s/it]

Kaggle Inference: 100%|██████████| 1000/1000 [7:19:04<00:00, 26.34s/it]

✅ Saved to submissions/submission_glm4.1v-9b-thinking_zero_shot_pd.csv


In [10]:
import torch
import gc
# Delete model and tokenizer from memory
del model
del tokenizer
if 'inferencer' in globals(): del inferencer
# Force garbage collection and clear CUDA cache
gc.collect()
torch.cuda.empty_cache()